# 🩺 Health Chatbot with RAG and Gemini — Itabira, Brazil

This notebook implements a chatbot that answers questions about **Covid-19** and about the **public healthcare network of Itabira (Minas Gerais, Brazil)**, with a graphical chat interface that runs inside Colab itself.

> 🇧🇷 **Language note:** the notebook's documentation and code are in English, but the chatbot itself operates in **Portuguese**. The knowledge base, the system instruction, the few-shot examples and the chat interface are intentionally kept in Portuguese, since they target Brazilian users and come from official Brazilian sources.

## The problem RAG solves

Large Language Models (**LLMs**) such as Gemini are trained on huge volumes of public text. They write very well, but they have two serious limitations for a health assistant:

- **They don't know local, specific information**, such as which neighborhoods belong to each region of Itabira or the phone number of each health unit (UBS).
- **When they don't know something, they may make it up**, producing a plausible and convincing answer. This phenomenon is called **hallucination**, and it is unacceptable when the answer tells someone where to seek medical care.

**RAG** (*Retrieval-Augmented Generation*) addresses this by splitting two responsibilities:

- the **knowledge** lives in a document base that we control, built from official sources;
- the **LLM** does not answer "from memory": it receives the relevant excerpt from the base and only organizes it into a clear answer.

We say the answer is **grounded** in the document. If the information changes, we only need to update the base, without retraining any model.

## How the pipeline works

```
 User question
        │
        ▼
 [1] Question embedding ──► [2] Vector search over the base ──► most similar document
                                                                      │
                                                                      ▼
                          [3] Prompt = instruction + examples + history + question + document
                                                                      │
                                                                      ▼
                                                   [4] Gemini LLM writes the answer
```

1. **Indexing** (done only once): each document in the base is converted into a numeric vector, the *embedding*, which represents its meaning.
2. **Retrieval**: the question is also turned into a vector, which is compared with the vectors in the base to find the document with the closest meaning.
3. **Augmentation**: the retrieved document is inserted into the prompt, together with the question and the answering rules.
4. **Generation**: Gemini reads all of this and produces a bullet-point answer that cites the sources.

## AI concepts applied

| Concept | What it is | Section |
|---|---|---|
| Semantic embeddings | Representing the meaning of a text as a vector | 5 |
| Vector search and cosine similarity | Finding the document most similar to the question | 5 |
| Relevance threshold | Rejecting questions the base doesn't cover | 5 |
| System instruction and *few-shot prompting* | Defining the model's role, format and limits through examples | 6 |
| Sampling parameters | Controlling how creative the generated text is | 6 |
| Safety filters | Blocking harmful content | 6 |
| Conversation memory | Giving the model the context of previous turns | 7 |
| Conversational interface | A graphical chat for the end user | 8 |

In the code cells, comments marked with **🧠** point out exactly where each concept is applied.

## How to use

1. Add your Gemini API key to **Secrets** (🔑 icon in the Colab sidebar) under the name `CHAVE_SECRETA` and grant this notebook access to it.
2. Run all cells in order (**Runtime → Run all**).
3. The chat window appears at the end, in section 8.

> ⚠️ **Disclaimer:** although the information comes from official channels, this chatbot is a **prototype** and **must not be used in real situations**. If you have symptoms, seek a healthcare facility.

## 1. Installing dependencies

Two libraries are required:

- **`google-genai`**: Google's official SDK for the Gemini API. It provides access to both **text generation** models and **embedding** models, and replaces the older `google-generativeai` library.
- **`gradio`**: a library for building web interfaces for AI models with just a few lines of code. It draws the chat window in section 8.

The installation intentionally does **not** use the `-U` (*upgrade*) flag. Colab ships with many pre-installed, interdependent Google libraries; forcing the latest version of one of them often breaks the version requirements of the others, producing warnings such as `pip's dependency resolver does not currently take into account...`. Without `-U`, pip only installs what is missing and keeps the environment consistent.

If a dependency conflict warning still shows up, it does **not** stop the installation. Only act if a later cell fails; in that case, use **Runtime → Disconnect and delete runtime** and run everything again.

In [ ]:
import sys
print("Python version")
print(sys.version)

%pip install -q google-genai gradio

## 2. Imports and configuration

All tunable parameters of the project live in this cell. To try a different model, make the search stricter or looser, or change the size of the chat memory, there is no need to hunt for values scattered across the code.

| Parameter | Purpose |
|---|---|
| `EMBEDDING_MODEL` | Model that turns text into vectors. It must be the **same** for indexing and querying, otherwise the vectors are not comparable. |
| `GENERATION_MODELS` | Models that write the answers, in order of preference. If the first one is overloaded, the next one is used. |
| `SIMILARITY_THRESHOLD` | Minimum score (from 0 to 1) for a document to be considered relevant. Higher values make the chatbot more cautious; lower values make it more permissive. |
| `MAX_RETRIES` and `INITIAL_BACKOFF_S` | How many times to retry a call that failed due to overload, and how long to wait between attempts. |
| `TRANSIENT_ERROR_CODES` | HTTP errors that indicate a temporary problem: `429` (rate limit) and `500`/`503`/`504` (server unavailable). |
| `MAX_HISTORY_TURNS` | How many previous exchanges (question + answer) the model receives as conversation memory. |
| `SHARE_LINK` | If `True`, Gradio also creates a temporary public link to the chat (see section 8). |

The API key is read from Colab's **Secrets** vault with `userdata.get`. It should never be written directly in the code, because notebooks are often shared and the key grants access to your usage quota.

In [ ]:
import inspect
import time

import gradio as gr
import numpy as np
import pandas as pd
from google import genai
from google.colab import userdata
from google.genai import errors, types

EMBEDDING_MODEL = "gemini-embedding-001"  # 🧠 Embedding model: converts text into meaning vectors

GENERATION_MODELS = [  # 🧠 Generative LLMs: preference list used for model fallback
    "models/gemini-3.5-flash-lite",
    "models/gemini-3.7-flash",
    "models/gemini-3.6-flash",
    "models/gemini-3.1-flash-lite",
    "models/gemini-3.8-flash",
]

SIMILARITY_THRESHOLD = 0.55  # 🧠 Relevance threshold: minimum cosine similarity to accept a document

MAX_RETRIES = 3
INITIAL_BACKOFF_S = 2
TRANSIENT_ERROR_CODES = {429, 500, 503, 504}

MAX_HISTORY_TURNS = 6  # 🧠 Context window: previous turns sent to the LLM as memory
SHARE_LINK = False

client = genai.Client(api_key=userdata.get("CHAVE_SECRETA"))

## 3. Available models

The Gemini API model catalog changes over time: new models are released, old ones are retired, and some are only available to certain accounts or regions. Each model reports the **actions it supports**; the two we care about are:

- `generateContent`: the model generates text, i.e. it is a **generative LLM**;
- `embedContent`: the model produces **embeddings**.

This cell queries the catalog **only once**, lists the models of each type and cross-checks the result against `GENERATION_MODELS`. Only the models from your list that actually exist for your key are used (`ACTIVE_MODELS`). If none of them exist, the notebook stops here with a clear message instead of failing later with a hard-to-understand error; just copy names from the printed list into `GENERATION_MODELS`.

In [ ]:
actions_by_model = {m.name: (m.supported_actions or []) for m in client.models.list()}

for label, action in [("Content generation", "generateContent"), ("Embeddings", "embedContent")]:
    print(f"\n--- {label} ---")
    for name, actions in actions_by_model.items():
        if action in actions:
            print(name)

ACTIVE_MODELS = [m for m in GENERATION_MODELS if "generateContent" in actions_by_model.get(m, [])]
if not ACTIVE_MODELS:
    raise RuntimeError("None of the GENERATION_MODELS is available. Update the list with names printed above.")

print("\nGeneration models in use (order of preference):", ACTIVE_MODELS)

## 4. Knowledge base

The knowledge base is the chatbot's **single source of truth**: everything it answers must be written here. It consists of five documents in Portuguese, each with:

- **`title`**: a summary of the topic. It is sent together with the text when generating the embedding and helps the model capture the document's main subject.
- **`content`**: the full text, which is handed to the LLM as the source when the document is retrieved.

### Why document size matters

In RAG, each document is a **retrieval unit** (a *chunk*): the search always returns a whole document, never part of it. This creates a trade-off:

- **small, focused** documents covering a single topic produce more precise embeddings and are easier to find;
- **large documents, or ones that mix several topics**, produce an "averaged" embedding that is less similar to any specific question.

Document 5 (neighborhoods and health units) is much larger than the others. It works because it covers a single topic, but if the base grows, a good improvement would be to split it into one document per region.

### How to extend the base

Create a `DOCUMENT6` in the same format and add it to the list passed to `pd.DataFrame`. Section 5 generates embeddings for all documents automatically. In production, with hundreds or thousands of documents, the index would move out of the notebook's memory into a **vector database**, such as Supabase's `pgvector`.

In [ ]:
# 🧠 Knowledge base: each document is a chunk that RAG can retrieve
DOCUMENT1 = {
    "title": "O que é o novo coronavírus, a Covid-19?",
    "content": "A COVID-19 é uma doença causada pelo coronavírus, denominado SARS-CoV-2, que apresenta um espectro clínico variando de infecções assintomáticas a quadros graves. De acordo com a Organização Mundial de Saúde, a maioria (cerca de 80%) dos pacientes com COVID-19 podem ser assintomáticos ou oligossintomáticos (poucos sintomas), e aproximadamente 20% dos casos detectados requer atendimento hospitalar por apresentarem dificuldade respiratória, dos quais aproximadamente 5% podem necessitar de suporte ventilatório. Fontes: Secretaria de Estado de Saúde de Minas Gerais e Organização Mundial de Saúde (OMS)."
}

DOCUMENT2 = {
    "title": "Quais são os sintomas mais comuns da Covid-19?",
    "content": "A COVID-19 (novo coronavírus) possui múltiplos sintomas, em que esses sintomas podem variar, sendo os mais comuns deles: resfriado, febre, sensação febril, tosse, coriza, dor de garganta, dor de cabeça, dificuldade para respirar, perda de olfato, alteração do paladar, náuseas, vômitos, diarreia, cansaço, diminuição do apetite e falta de ar. Fonte: Secretaria de Estado de Saúde de Minas Gerais."
}

DOCUMENT3 = {
    "title": "O que fazer caso esteja com sintomas?",
    "content": "Se você estiver doente, com sintomas compatíveis com a COVID-19, tais como febre, tosse, dor de garganta e/ou coriza, com ou sem falta de ar, evite contato físico com outras pessoas, incluindo os familiares, principalmente, idosos e doentes crônicos, procure imediatamente os postos de triagem nas Unidades Básica de Saúde (UBS) ou outras unidades de saúde. Fonte: Secretaria de Estado de Saúde de Minas Gerais."
}

DOCUMENT4 = {
    "title": "Grupos de risco da Covid-19",
    "content": "Alguns grupos têm maior chance de desenvolver as formas mais graves da doença covid-19, como: idosos (idade igual ou superior a 60 anos), pessoas com doenças cardíacas, pessoas com doenças pulmonares, como asma e Doença Pulmonar Obstrutiva Crônica, pessoas com problemas de baixa imunidade, como pessoas transplantadas ou em quimioterapia, pessoas com doenças renais ou em diálise, diabéticos, gestantes de alto risco, pessoas com doenças do fígado e obesos (IMC>=40). Fonte: Secretaria de Estado de Saúde de Minas Gerais."
}

DOCUMENT5 = {
    "title": "Bairros e Unidades de Saúde da Cidade de Itabira Separados por Região",
    "content": "Bairros da Região 1: Fênix, João XXIII, Valença, Monsenhor José Lopes, Santa Marta, Santa Ruth, Ribeira de Cima, Pedras do Vale, Jardim Belvedere, Bálsamos, Conceição, Vila Técnica Conceição, Abóboras, Barreiro, Condomínio Retiro da Serra, Cedro, Jardim Universitário, CDI I e CDI II. Unidades de Saúde da Região 1: UBS Fênix (3839-2489), UBS João XXIII (3839-2400), UBS Santa Marta (3839-2485), UBS Santa Ruth (3839-2485) e UBS Barreiro (3839-2788). Bairros da Região 2: Gabiroba, Ribeira de Baixo, Santa Tereza, Jardins dos Ipês, Bethânia, Fazenda do Lago, Flamboyant, Vale do Sol, Loteamento da Cohapi e Candidópolis. Unidades de Saúde da Região 2: UBS Bethânia (3839-2935), UBS Gabiroba de Baixo (3839-2491) e UBS Gabiroba de Cima (3839-2492). Bairros da Região 3: Amazonas, Novo Amazonas, Clóvis Alvim I, Clóvis Alvim II, Hamilton, Gianetti, Major Lage (Areão), Caminho Novo, São Tomé, São Bento, Cônego Guilhermino, Vila Dom Prudêncio, São Marcos, Panorama, Água Fresca, Vila Santa Bárbara, Vila Salica, Madre Maria de Jesus, Machado, Juca Batista e Loteamento José Leandro de Araújo. Unidades de Saúde da Região 3: UBS Amazonas (3839-2225), UBS Clóvis Alvim (3839-2165), UBS Major Lage (3839-2324), UBS Areão (3839-2482), UBS Água Fresca (3839-2222), UBS Machado (3839-2497) e UBS Juca Batista (3839-2495). Bairros da Região 4: Colina da Praia, Praia, Eldorado, São Francisco, Cidade Nova, Santa Inês, Santa Matilde, João Fortunato, Alvorada, Juca Rosa, Nossa Senhora das Oliveiras, São Cristóvão, Alta Vista, Belo Monte, Jardim Buritis, Jardim Belvedere, Jardim Mangabeiras, Condomínio Beleville, Condomínio Morada da Brisa e Condomínio Village da Lagoa. Unidades de Saúde da Região 4: UBS Praia I (3831-3819), UBS Praia II (3839-2224), UBS Eldorado (3839-2833), UBS Juca Rosa (3839-2496) e UBS Nossa Senhora das Oliveiras (3839-2574). Bairros da Região 5: Pedreira. Unidades de Saúde da Região 5: UBS Pedreira I (3839-2484) e UBS Pedreira II (3839-2583). Bairros da Região 6: Centro, Penha, Campestre, Bela Vista, Nova Vista, São Pedro, Vila Santa Izabel, Vila Santa Rosa, Vila Piedade, Esplanada da Estação, 14 de Fevereiro, Vila São Joaquim, Alto Pereira, Vila Amélia, Moinho Velho, Chacrinha, Vila Paciência, Vila das Nações Unidas, Santo Antônio e Pará. Unidades de Saúde da Região 6: Hospital Nossa Senhora das Dores (3839-1405), UBS Centro (3839-2488), UBS Campestre (3839-2487), UBS Bela Vista (3839-2486), UBS Nova Vista (3839-2499), UBS Vila Santa Rosa (3839-2494), UBS Vila Piedade (3839-2493) e UBS Pará (3839-2483). Bairros da Região 7: Chapada, Boa Esperança e Barro Branco. Unidades de Saúde da Região 7: UBS Chapada (3839-2927). Bairros da Região 8: Senhora do Carmo. Unidades de Saúde da Região 8: UBS Senhora do Carmo (3839-2766). Bairros da Região 9: Ipoema. Unidades de Saúde da Região 9: UBS Ipoema (3839-2780). Fontes: Secretaria Municipal de Saúde de Itabira e Prefeitura de Itabira."
}

knowledge_base = pd.DataFrame([DOCUMENT1, DOCUMENT2, DOCUMENT3, DOCUMENT4, DOCUMENT5])
knowledge_base[["title"]]

## 5. Embeddings and semantic search

This is the **retrieval** stage of RAG: given a question, find which document in the base covers the same subject.

### What an embedding is

An **embedding** is a list of numbers (a vector) that represents the **meaning** of a text. The `gemini-embedding-001` model produces vectors with 3,072 numbers. Texts with similar meanings produce vectors pointing in similar directions, even when they use different words.

This is what sets **semantic search** apart from keyword search. The question *"o que fazer se eu estiver com falta de ar?"* ("what should I do if I'm short of breath?") finds the document that mentions *"dificuldade para respirar"* ("difficulty breathing"), even though the two phrases share almost no words.

### Asymmetric embeddings: document × query

Questions and documents look different: questions are short and interrogative, documents are long and declarative. That's why the API offers distinct **task types**, each optimizing the vector for its role:

- `RETRIEVAL_DOCUMENT`: used when **indexing** the base documents (it also accepts the title);
- `RETRIEVAL_QUERY`: used for the user's **questions**.

### Cosine similarity

To measure how alike two vectors are, we use **cosine similarity**, which measures the angle between them: the closer to 1, the more similar the meanings. If all vectors are **normalized** (scaled to length 1), cosine similarity equals a simple dot product. That's why `embed_text` normalizes every vector, and the search compares the question against **all** documents at once by multiplying the embedding matrix by the question vector.

### Relevance threshold

The search always finds *some* most-similar document, even for questions unrelated to the base, such as *"qual é a capital da França?"* ("what is the capital of France?"). Without a filter, the chatbot would hand an irrelevant text to the LLM, and the model would try to answer from it. `SIMILARITY_THRESHOLD` sets a minimum score: below it, we consider that the base **does not cover** the subject, and the chatbot says so instead of risking an answer.

### Retry with exponential backoff

The API may temporarily reject calls due to overload or too many requests. The `with_retry` function retries the call in those cases, doubling the wait after each attempt (2 s, 4 s, ...) so as not to worsen the overload. Non-transient errors, such as an invalid key, are raised immediately, because retrying would not help.

In [ ]:
def with_retry(func, *args, **kwargs):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return func(*args, **kwargs)
        except errors.APIError as error:
            if error.code not in TRANSIENT_ERROR_CODES or attempt == MAX_RETRIES:
                raise
            wait = INITIAL_BACKOFF_S * 2 ** (attempt - 1)
            print(f"[Error {error.code}] Retrying in {wait}s ({attempt}/{MAX_RETRIES})...")
            time.sleep(wait)


def embed_text(text, task_type, title=None):
    response = with_retry(
        client.models.embed_content,
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type, title=title),  # 🧠 Asymmetric embedding: different task types for documents and queries
    )
    vector = np.asarray(response.embeddings[0].values, dtype=np.float32)
    return vector / np.linalg.norm(vector)  # 🧠 L2 normalization: the dot product becomes cosine similarity


def retrieve_document(question):
    similarities = EMBEDDING_MATRIX @ embed_text(question, "RETRIEVAL_QUERY")  # 🧠 Vector search: compares the question with every document
    index = int(np.argmax(similarities))  # 🧠 Top-1 retrieval: picks the most similar document
    similarity = float(similarities[index])
    document = knowledge_base.iloc[index] if similarity >= SIMILARITY_THRESHOLD else None  # 🧠 Relevance threshold: discards unrelated context
    return document, similarity


EMBEDDING_MATRIX = np.stack([  # 🧠 Indexing: the base becomes a vector index (documents × dimensions)
    embed_text(doc.content, "RETRIEVAL_DOCUMENT", title=doc.title)
    for doc in knowledge_base.itertuples()
])
print("Indexed knowledge base:", EMBEDDING_MATRIX.shape)

### Testing and calibrating the search

Before plugging in the LLM, it's worth checking whether retrieval picks the right documents. The cell below shows, for each test question (in Portuguese, like the base), the **similarity score** and the selected document.

Use the scores to calibrate `SIMILARITY_THRESHOLD` (section 2). The ideal value lies **between** the lowest score among the questions the base covers and the score of the last question, which is deliberately out of scope. If a valid question shows up as *"no relevant document"*, the threshold is too high; if the question about France finds a document, it is too low. After changing the value, run sections 2 and 5 again.

In [ ]:
test_questions = [
    "O que é COVID-19?",
    "Quais são os grupos de risco?",
    "Qual UBS atende o bairro Gabiroba?",
    "Qual é a capital da França?",
]

for question in test_questions:
    document, similarity = retrieve_document(question)
    result = document.title if document is not None else "(no relevant document)"
    print(f"{similarity:.3f} | {question} -> {result}")

## 6. Answer generation

This is the **generation** stage of RAG: the LLM receives the retrieved document and writes the answer. The quality of that answer depends less on the model itself and more on **how it is instructed and configured**.

### System instruction and *few-shot prompting*

The **system instruction** (`system_instruction`) is a text sent to the model on every call, taking precedence over user messages. It defines the chatbot's behavior (it is written in Portuguese, so the answers come out in Portuguese):

- **format**: answer in bullet points, highlighting the key information;
- **traceability**: always cite the sources of the base text;
- **anti-hallucination constraint**: add nothing that isn't in the base text. This rule is what makes the answer truly grounded in the document.

Besides the rules, the instruction contains **two complete question-and-answer examples**. This technique is called ***few-shot prompting***: instead of only describing the desired format, we show examples of it, and the model reproduces the structure much more consistently.

### Sampling parameters

The LLM generates text **one token (piece of a word) at a time**, sampling each one from the most likely candidates. Two parameters control this sampling:

- **`temperature`** (0.5): the lower it is, the more predictable and faithful to the base text; the higher, the more varied and "creative". An intermediate value preserves faithfulness without making the text robotic.
- **`top_p`** (0.9), also known as ***nucleus sampling***: sampling only considers the smallest set of tokens whose combined probability reaches 90%, discarding very unlikely options.

`max_output_tokens` caps the length of the answer, which also controls the cost of each call.

### Safety filters

**Safety filters** classify content into four harm categories (harassment, hate speech, sexually explicit content and dangerous content) and block answers with a **medium or high** probability of falling into any of them. When something is blocked, the model returns an empty answer, and the chatbot displays a default message instead of crashing.

### Model fallback

The `generate_answer` function walks through `ACTIVE_MODELS` in order. Each model gets a few attempts; if it remains unavailable, the next one in the list takes over. The chatbot only fails if **all** models are down at the same time.

In [ ]:
# 🧠 System instruction + few-shot prompting: defines role, format, sources and the ban on inventing information
SYSTEM_INSTRUCTION = """
Utilizando como base os exemplos de perguntas e respostas abaixo, responda à pergunta atual utilizando o texto base num formato organizado, separando as informações em tópicos para destaque, incluindo as fontes do texto base, mas sem adicionar informações que não fazem parte do texto base.

EXEMPLO DE PERGUNTA 1: O que é a Covid?
EXEMPLO DE RESPOSTA 1:
* **COVID-19:** Doença causada pelo coronavírus SARS-CoV-2.
* **Espectro Clínico:** De infecções assintomáticas a quadros graves.
* **Pacientes (OMS):**
  * ~80% assintomáticos ou oligossintomáticos (poucos sintomas).
  * ~20% requerem hospitalização (dificuldade respiratória).
  * ~5% dos hospitalizados necessitam de suporte ventilatório.
* **Fontes:** Secretaria de Estado de Saúde de Minas Gerais e Organização Mundial de Saúde (OMS).

EXEMPLO DE PERGUNTA 2: Quais são as UBS perto do bairro São Bento?
EXEMPLO DE RESPOSTA 2:
* **Região do Bairro São Bento:** Região 3
* **Unidades de Saúde da Região 3:**
  * UBS Amazonas (3839-2225)
  * UBS Clóvis Alvim (3839-2165)
  * UBS Major Lage (3839-2324)
  * UBS Areão (3839-2482)
  * UBS Água Fresca (3839-2222)
  * UBS Machado (3839-2497)
  * UBS Juca Batista (3839-2495)
* **Fontes:** Secretaria Municipal de Saúde de Itabira e Prefeitura de Itabira.
"""

SAFETY_SETTINGS = [  # 🧠 Safety filters: block harmful content with medium or high probability
    types.SafetySetting(category=category, threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE)
    for category in (
        types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
    )
]

GENERATION_CONFIG = types.GenerateContentConfig(
    temperature=0.5,  # 🧠 Temperature: controls randomness when choosing each token
    top_p=0.9,  # 🧠 Nucleus sampling: samples only from tokens adding up to 90% probability
    max_output_tokens=2048,
    safety_settings=SAFETY_SETTINGS,
    system_instruction=SYSTEM_INSTRUCTION,
)


def generate_answer(contents):
    last_error = None
    for model in ACTIVE_MODELS:
        try:
            response = with_retry(
                client.models.generate_content,  # 🧠 LLM inference: autoregressive generation of the answer
                model=model,
                contents=contents,
                config=GENERATION_CONFIG,
            )
            return response.text or "Não foi possível gerar uma resposta para essa pergunta."  # 🧠 Empty text means the safety filters blocked the answer
        except errors.APIError as error:
            if error.code not in TRANSIENT_ERROR_CODES:
                raise
            print(f"[{model} unavailable] Trying the next model...")
            last_error = error
    raise RuntimeError("Todos os modelos estão indisponíveis. Tente novamente em alguns minutos.") from last_error

## 7. Chatbot logic

The `answer` function is the **RAG orchestrator**: it receives the question typed in the interface and runs the retrieval, augmentation and generation stages in sequence.

### Augmentation: building the prompt

The retrieved document is inserted into the prompt in the format `PERGUNTA ATUAL: ... / TEXTO BASE: ...` ("current question" / "base text"), the same one used in the system instruction examples. Keeping this pattern helps the model recognize where the question is and where the source it must use is.

When the search finds no document above the threshold, the LLM **is not even called**: the chatbot replies directly that it doesn't have that information and lists the topics it knows about. This prevents hallucinations and saves an API call.

User-facing messages (such as the "no information found" reply and error messages shown in the chat) are kept in Portuguese, consistent with the rest of the conversation.

### Conversation memory

LLMs are **stateless** between calls: each request is independent. For the chatbot to understand follow-ups such as *"e quais são os sintomas?"* ("and what are the symptoms?"), previous turns must be sent again along with the new question. This set of messages forms the model's **context window**.

Two decisions keep this memory cheap:

- the history stores only **questions and answers**, without the base texts, which are long and already summarized in the answers;
- only the last `MAX_HISTORY_TURNS` turns are sent (a **sliding window**), so the cost per question doesn't grow indefinitely.

The history comes from the Gradio interface itself, and `convert_history` translates it into Gemini's message format, in which the chatbot has the `model` role and the user has the `user` role.

### Known limitation

The **search** uses only the current question, not the history. Very short follow-up questions, such as *"e os telefones?"* ("and the phone numbers?"), carry little meaning on their own and may fall below the threshold. A common improvement is **query rewriting**: asking the LLM to turn the question into a complete one, such as *"quais os telefones das UBS da região 3?"* ("what are the phone numbers of the health units in region 3?"), before running the search.

In [ ]:
NO_CONTEXT_MESSAGE = (
    "Não encontrei essa informação na minha base de conhecimento. "
    "Posso responder sobre a Covid-19 (definição, sintomas, o que fazer, grupos de risco) "
    "e sobre as unidades de saúde de Itabira por bairro."
)


def _message(role, text):
    return types.Content(role=role, parts=[types.Part(text=text)])


def _extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text") or ""
    if isinstance(content, (list, tuple)):
        return " ".join(_extract_text(part) for part in content)
    return ""


def convert_history(gradio_history):
    messages = []
    for item in gradio_history:
        if isinstance(item, dict):
            pairs = [(item.get("role"), item.get("content"))]
        else:
            pairs = [("user", item[0]), ("assistant", item[1])]
        for role, content in pairs:
            text = _extract_text(content).strip()
            if text and role in ("user", "assistant"):
                messages.append(_message("model" if role == "assistant" else "user", text))

    messages = messages[-2 * MAX_HISTORY_TURNS:]  # 🧠 Short-term memory: a sliding window caps the tokens sent
    while messages and messages[0].role == "model":
        messages.pop(0)
    return messages


def answer(question, gradio_history):
    question = (question or "").strip()
    if not question:
        return "Digite uma pergunta."
    try:
        document, _ = retrieve_document(question)  # 🧠 RAG, retrieval: finds the most relevant document
        if document is None:
            return NO_CONTEXT_MESSAGE
        prompt = f"PERGUNTA ATUAL: {question}\nTEXTO BASE: {document.content}"  # 🧠 RAG, augmentation: grounds the prompt in the document
        return generate_answer(convert_history(gradio_history) + [_message("user", prompt)])  # 🧠 RAG, generation: LLM answers with memory + context
    except (errors.APIError, RuntimeError) as error:
        return f"⚠️ Não foi possível responder agora. Detalhe: {error}"

## 8. Graphical interface (Gradio)

The interface uses Gradio's **`gr.ChatInterface`** component, which builds a complete chat window from a single function, `answer`. It takes care of:

- displaying messages as chat bubbles with Markdown support (the bullet points and bold text in the answers are rendered);
- keeping the conversation history and passing it to the function on every new question;
- showing clickable **example questions**, useful for first-time users.

The title, description and example questions shown in the window are in Portuguese, since the interface is meant for the chatbot's end users.

### How to use

- The chat appears right below the cell. Type your question and press **Enter**.
- To start the conversation over, use the chat's own clear button.
- To shut it down, interrupt the cell or disconnect the runtime. Running the cell again closes the previous interface before opening a new one.

### Public link

With `SHARE_LINK = True` (section 2), Gradio also creates a public address like `https://xxxx.gradio.live`, valid for about 72 hours and while the notebook is running. It's useful for demoing the prototype to other people, but **anyone with the link uses your Gemini API quota**, so only share it with people you trust.

### Version compatibility

Gradio versions differ in the chat history format: some use `[question, answer]` pairs and others use messages with `role` and `content`. The cell checks whether the installed version accepts the `type` parameter and, if so, requests the message format. The `convert_history` function in section 7 accepts both formats.

In [ ]:
if "chat_interface" in globals():
    chat_interface.close()

version_kwargs = {"type": "messages"} if "type" in inspect.signature(gr.ChatInterface).parameters else {}

chat_interface = gr.ChatInterface(  # 🧠 Conversational interface: chat that keeps the history and feeds it to RAG on every turn
    fn=answer,
    title="🩺 Chatbot de Saúde — Itabira/MG",
    description=(
        "Pergunte sobre a Covid-19 ou sobre as unidades de saúde de Itabira. "
        "As respostas são geradas por IA a partir de fontes oficiais.<br>"
        "⚠️ <b>Protótipo:</b> não utilize em situação real. Em caso de sintomas, procure uma unidade de saúde."
    ),
    examples=[
        "O que é a Covid-19?",
        "Quais são os sintomas mais comuns?",
        "O que devo fazer se estiver com sintomas?",
        "Quais são os grupos de risco?",
        "Quais são as UBS perto do bairro São Bento?",
    ],
    **version_kwargs,
)

chat_interface.launch(share=SHARE_LINK)